# 视频分类 C3D

In [27]:
import os
import cv2
import random
import numpy as np
from tqdm import tqdm
from pathlib import Path

import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.models as models
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score


## 一、处理数据

In [28]:
ucf_root_dir = r"/mnt/Data/daihongliang"
ucf_dataset_name = "数据集-clean"
ucf_processed_name = "shop-Processed"
ucf_train_test_file_dir = "shopTrainTestlist"

In [29]:
def get_train_test_lists(ucf_root_dir, ucf_dataset_name):
    data_dir = os.path.join(ucf_root_dir, ucf_dataset_name)
    class_list = os.listdir(data_dir)
    data_list = []

    for i in range(len(class_list)):
        class_item_dir = os.path.join(data_dir, class_list[i])
        data_list.extend([os.path.join(class_item_dir, it) for it in os.listdir(class_item_dir)])

    train_list, test_list = train_test_split(data_list, test_size=0.2)

    return train_list, test_list

In [30]:
train_data, test_data = get_train_test_lists(ucf_root_dir, ucf_dataset_name)

In [31]:
train_data[:5]

['/mnt/Data/daihongliang/数据集-clean/hiding/hiding (68).mp4',
 '/mnt/Data/daihongliang/数据集-clean/hiding/hiding (18).mp4',
 '/mnt/Data/daihongliang/数据集-clean/eating/video13.mp4',
 '/mnt/Data/daihongliang/数据集-clean/hiding/hiding (63).mp4',
 '/mnt/Data/daihongliang/数据集-clean/eating/eating (39).mp4']

In [32]:
test_data[:5]

['/mnt/Data/daihongliang/数据集-clean/eating/video-15.mp4',
 '/mnt/Data/daihongliang/数据集-clean/tearing/video6.mp4',
 '/mnt/Data/daihongliang/数据集-clean/drinking/drinking (9).mp4',
 '/mnt/Data/daihongliang/数据集-clean/drinking/video-1.mp4',
 '/mnt/Data/daihongliang/数据集-clean/eating/eating_03_14-1.mp4']

## 二、编写 Torch 的 Dataset

In [33]:
class UCFDataset(Dataset):
    def __init__(self, ucf_root_dir, ucf_dataset_name, ucf_train_test_file_dir, train=False):
        # 读取训练标签
        self.classes = {}
        classes_file = os.path.join(ucf_root_dir, ucf_train_test_file_dir, 'classshop.txt')
        with open(classes_file) as fin:
            for row in fin:
                line_info = row.strip().split(' ')
                self.classes[line_info[1]] = int(line_info[0]) - 1

        # 读取数据
        train_data, test_data = get_train_test_lists(ucf_root_dir, ucf_dataset_name)
        self.data = train_data if train else test_data

    def __len__(self):
        return len(self.data)

    def __getitem__(self, i):
        cap = cv2.VideoCapture(self.data[i])

        frames = []
        while True:
            ret, frame = cap.read()
            if not ret:
                break
            frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)  # 转换BGR到RGB
            frame_resized = cv2.resize(frame_rgb, (112, 112))  # 调整尺寸到目标大小
            frame_normalized = frame_resized.astype(np.float32) # / 255.0  # 归一化到 [0, 1] 范围
            # mean = [0.45, 0.45, 0.45]  # 标准归一化均值[3](@ref)
            # std = [0.225, 0.225, 0.225]  # 标准归一化标准差[3](@ref)
            # frame_normalized = (frame_normalized - mean) / std  # 应用归一化
            frame_chw = np.transpose(frame_normalized, (2, 0, 1))  # 转换维度: (H, W, C) -> (C, H, W) 符合PyTorch格式
            frames.append(frame_chw)
        cap.release()
        frames_tensor = torch.tensor(np.stack(frames), dtype=torch.float32)

        return frames_tensor, self.classes[self.data[i].split('/')[-2]]

In [34]:
trainset = UCFDataset(ucf_root_dir, ucf_dataset_name, ucf_train_test_file_dir, train=True)
testset = UCFDataset(ucf_root_dir, ucf_dataset_name, ucf_train_test_file_dir, train=False)

In [35]:
d, l = trainset.__getitem__(123)

## 三、编写 Torch 的 DataLoader

In [36]:
def collate_fn(batch):
    data = []
    label = []
    
    # 筛选总帧数小于48的样本
    for d, l in batch:
        num_frames = d.shape[0]
        if num_frames < 64:
            continue  # 跳过帧数不足的样本
        
        # 随机选取一个48帧的连续片段
        if num_frames > 64:
            start_idx = random.randint(0, num_frames - 64)
            d = d[start_idx: start_idx + 64]  # 提取48帧片段
        data.append(d)
        label.append(l)
    
    # 处理批量中无有效样本的情况
    if len(data) == 0:
        return torch.tensor([]), torch.tensor([])
    
    # 按照1/3比例采样，随机选择起始点(0,1,2)
    start_index = random.randint(0, 3)  # 随机起始帧[2]
    sampled_data = []
    for d in data:
        sampled_d = d[start_index::4]  # 从随机点开始每3帧取1帧
        sampled_data.append(sampled_d)
    
    # 组装张量（所有样本采样后帧数相同）
    data_tensor = torch.stack(sampled_data, dim=0).type(torch.FloatTensor).transpose(1, 2).contiguous()
    label_tensor = torch.tensor(label).type(torch.FloatTensor)
    
    return data_tensor, label_tensor

In [37]:
trainloader = DataLoader(trainset, batch_size=8, shuffle=True, collate_fn=collate_fn, num_workers=8)
testloader = DataLoader(testset, batch_size=8, shuffle=True, collate_fn=collate_fn, num_workers=8)

batch_data, batch_labels = next(iter(trainloader))
print("数据形状:", batch_data.shape)
print("标签形状:", batch_labels.shape)

数据形状: torch.Size([7, 3, 16, 112, 112])
标签形状: torch.Size([7])


## 四、设计网络

In [38]:
class C3D(nn.Module):

    def __init__(self, num_classes):
        super(C3D, self).__init__()

        self.conv1 = nn.Conv3d(3, 64, kernel_size=(3, 3, 3), padding=(1, 1, 1))
        self.pool1 = nn.MaxPool3d(kernel_size=(1, 2, 2), stride=(1, 2, 2))

        self.conv2 = nn.Conv3d(64, 128, kernel_size=(3, 3, 3), padding=(1, 1, 1))
        self.pool2 = nn.MaxPool3d(kernel_size=(2, 2, 2), stride=(2, 2, 2))

        self.conv3a = nn.Conv3d(128, 256, kernel_size=(3, 3, 3), padding=(1, 1, 1))
        self.conv3b = nn.Conv3d(256, 256, kernel_size=(3, 3, 3), padding=(1, 1, 1))
        self.pool3 = nn.MaxPool3d(kernel_size=(2, 2, 2), stride=(2, 2, 2))

        self.conv4a = nn.Conv3d(256, 512, kernel_size=(3, 3, 3), padding=(1, 1, 1))
        self.conv4b = nn.Conv3d(512, 512, kernel_size=(3, 3, 3), padding=(1, 1, 1))
        self.pool4 = nn.MaxPool3d(kernel_size=(2, 2, 2), stride=(2, 2, 2))

        self.conv5a = nn.Conv3d(512, 512, kernel_size=(3, 3, 3), padding=(1, 1, 1))
        self.conv5b = nn.Conv3d(512, 512, kernel_size=(3, 3, 3), padding=(1, 1, 1))
        self.pool5 = nn.MaxPool3d(kernel_size=(2, 2, 2), stride=(2, 2, 2), padding=(0, 1, 1))

        self.fc6 = nn.Linear(8192, 4096)
        self.fc7 = nn.Linear(4096, 4096)
        self.fc8 = nn.Linear(4096, num_classes)

        self.dropout = nn.Dropout(p=0.5)

        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.relu(self.conv1(x))
        x = self.pool1(x)
        x = self.relu(self.conv2(x))
        x = self.pool2(x)
        x = self.relu(self.conv3a(x))
        x = self.relu(self.conv3b(x))
        x = self.pool3(x)
        x = self.relu(self.conv4a(x))
        x = self.relu(self.conv4b(x))
        x = self.pool4(x)
        x = self.relu(self.conv5a(x))
        x = self.relu(self.conv5b(x))
        x = self.pool5(x)
        x = x.view(-1, 8192)
        x = self.relu(self.fc6(x))
        x = self.dropout(x)
        x = self.relu(self.fc7(x))
        x = self.dropout(x)
        logits = self.fc8(x)

        return logits

In [39]:
net = C3D(4)
o = net(batch_data)
print(o.shape)

torch.Size([7, 4])


## 五、构建训练代码

In [40]:
seed = 1
torch.manual_seed(seed)  # 为CPU设置随机种子
torch.cuda.manual_seed(seed)  # 为当前GPU设置随机种子
torch.cuda.manual_seed_all(seed)  # if you are using multi-GPU，为所有GPU设置随机种子
np.random.seed(seed)  # Numpy module.
random.seed(seed)  # Python random module.
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True

In [41]:
save_root = "./checkpoints_shop"
num_classes = 4
learning_rate = 1e-5
train_epochs = 100

In [42]:
# 创建检查目录
os.makedirs(os.path.join(save_root, 'acc'), exist_ok=True)
os.makedirs(os.path.join(save_root, 'loss'), exist_ok=True)

In [43]:
model = C3D(num_classes)
if torch.cuda.device_count() > 1:
    print(f"使用 {torch.cuda.device_count()} 个GPU进行训练。")
    model = nn.DataParallel(model)  # 包装模型
model = model.cuda()

使用 4 个GPU进行训练。


In [44]:
optimizer = optim.Adam(model.parameters(), lr=learning_rate)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=50, gamma=0.1)
criterion = nn.CrossEntropyLoss()

In [45]:
def calculate_metrics(y_true, y_pred):
    """计算各种评估指标"""
    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, average='weighted', zero_division=0)
    recall = recall_score(y_true, y_pred, average='weighted', zero_division=0)
    micro_f1 = f1_score(y_true, y_pred, average='micro', zero_division=0)
    macro_f1 = f1_score(y_true, y_pred, average='macro', zero_division=0)
    
    return accuracy, precision, recall, micro_f1, macro_f1

In [46]:
best_accuracy = 0
min_loss = float('inf')

for epoch in range(1, train_epochs + 1):
    print(f"\n{'='*60}")
    print(f"Epoch {epoch}/{train_epochs}")
    print(f"{'='*60}")
    
    # 更新学习率
    scheduler.step()
    current_lr = optimizer.param_groups[0]['lr']
    print(f"当前学习率: {current_lr}")

    # 训练循环
    model.train()
    train_loss, train_correct, train_num = 0.0, 0, 0
    train_pred, train_label = [] , []
    for _, (x, y) in enumerate(tqdm(trainloader, desc="训练中")):
        if not x.shape[0]:
            continue
        x, y = x.cuda(), y.long().cuda()
        
        # 梯度清零
        optimizer.zero_grad()
        
        # 前向传播
        outputs = model(x)
        loss = criterion(outputs, y)
        
        # 反向传播
        loss.backward()
        optimizer.step()
        
        # 统计信息
        train_loss += loss.item()
        _, predicted = torch.max(outputs.data, 1)
        train_correct += (predicted == y).sum().item()
        train_num += y.size(0)
        
        train_pred.extend(predicted.cpu().numpy())
        train_label.extend(y.cpu().numpy())

    train_avg_loss = train_loss / len(trainloader)
    train_accuracy, precision, recall, micro_f1, macro_f1 = calculate_metrics(train_label, train_pred)
    print(f"Loss: {train_avg_loss:.4f}, Accuracy: {train_accuracy:.2%}, Precision: {precision:.4f}, Recall: {recall:.4f}, Micro F1: {micro_f1:.4f}, Macro F1: {macro_f1:.4f}")

    # 测试循环
    model.eval()
    test_loss, test_correct_num, test_num = 0.0, 0, 0
    test_pred, test_label = [], []
    
    with torch.no_grad():
        for _, (x, y) in enumerate(tqdm(testloader, desc="测试中")):
            if not x.shape[0]:
                continue
            x, y = x.cuda(), y.long().cuda()
            
            outputs = model(x)
            loss = criterion(outputs, y)
            
            test_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            test_correct_num += (predicted == y).sum().item()
            test_num += y.size(0)
            
            test_pred.extend(predicted.cpu().numpy())
            test_label.extend(y.cpu().numpy())
    
    test_avg_loss = test_loss / len(testloader)
    test_accuracy, precision, recall, micro_f1, macro_f1 = calculate_metrics(test_label, test_pred)
    print(f"Loss: {test_avg_loss:.4f}, Accuracy: {test_accuracy:.2%}, Precision: {precision:.4f}, Recall: {recall:.4f}, Micro F1: {micro_f1:.4f}, Macro F1: {macro_f1:.4f}")

     # 保存最佳准确率模型
    if test_accuracy > best_accuracy:
        best_accuracy = test_accuracy
        save_path = os.path.join(save_root, 'acc', f'best_acc.pth')
        torch.save(model.state_dict(), save_path)
        print(f"模型已保存: {save_path} (Epoch {epoch}, Acc: {test_accuracy:.2%}, Loss: {test_avg_loss:.4f})")
    
    # 保存最低损失模型
    if test_avg_loss < min_loss:
        min_loss = test_avg_loss
        save_path = os.path.join(save_root, 'loss', f'best_loss.pth')
        torch.save(model.state_dict(), save_path)
        print(f"模型已保存: {save_path} (Epoch {epoch}, Acc: {test_accuracy:.2%}, Loss: {test_avg_loss:.4f})")

    
    # 正确率超过99%时结束训练
    if test_accuracy >= 0.99:
        print(f"训练准确率已达到 {test_accuracy:.2%}，提前终止训练")
        save_path = os.path.join(save_root, f'early_stop_epoch_{epoch}.pth')
        torch.save(model.state_dict(), save_path)
        print(f"模型已保存: {save_path} (Epoch {epoch}, Acc: {test_accuracy:.2%}, Loss: {test_avg_loss:.4f})")
        break


/home/wuhao@kemande.cn/anaconda3/envs/C3d_new/lib/python3.8/site-packages/torch/optim/lr_scheduler.py:131: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  warnings.warn("Detected call of `lr_scheduler.step()` before `optimizer.step()`. "



Epoch 1/100
当前学习率: 1e-05


训练中: 100%|██████████| 67/67 [00:26<00:00,  2.51it/s]


Loss: 1.3803, Accuracy: 31.51%, Precision: 0.2157, Recall: 0.3151, Micro F1: 0.3151, Macro F1: 0.2032


测试中: 100%|██████████| 17/17 [00:07<00:00,  2.14it/s]


...

Loss: 0.9031, Accuracy: 63.93%, Precision: 0.6381, Recall: 0.6393, Micro F1: 0.6393, Macro F1: 0.6089

Epoch 100/100
当前学习率: 1.0000000000000002e-07


训练中: 100%|██████████| 67/67 [00:24<00:00,  2.75it/s]


Loss: 0.9055, Accuracy: 61.13%, Precision: 0.6009, Recall: 0.6113, Micro F1: 0.6113, Macro F1: 0.5856


测试中: 100%|██████████| 17/17 [00:08<00:00,  2.00it/s]

Loss: 0.9287, Accuracy: 60.66%, Precision: 0.6185, Recall: 0.6066, Micro F1: 0.6066, Macro F1: 0.5848


In [ ]:
import torch
import numpy as np
import cv2
import os
import argparse
from pathlib import Path
import torch.nn as nn


# 定义C3D模型（与训练时相同的结构）
class C3D(nn.Module):
    def __init__(self, num_classes):
        super(C3D, self).__init__()

        self.conv1 = nn.Conv3d(3, 64, kernel_size=(3, 3, 3), padding=(1, 1, 1))
        self.pool1 = nn.MaxPool3d(kernel_size=(1, 2, 2), stride=(1, 2, 2))

        self.conv2 = nn.Conv3d(64, 128, kernel_size=(3, 3, 3), padding=(1, 1, 1))
        self.pool2 = nn.MaxPool3d(kernel_size=(2, 2, 2), stride=(2, 2, 2))

        self.conv3a = nn.Conv3d(128, 256, kernel_size=(3, 3, 3), padding=(1, 1, 1))
        self.conv3b = nn.Conv3d(256, 256, kernel_size=(3, 3, 3), padding=(1, 1, 1))
        self.pool3 = nn.MaxPool3d(kernel_size=(2, 2, 2), stride=(2, 2, 2))

        self.conv4a = nn.Conv3d(256, 512, kernel_size=(3, 3, 3), padding=(1, 1, 1))
        self.conv4b = nn.Conv3d(512, 512, kernel_size=(3, 3, 3), padding=(1, 1, 1))
        self.pool4 = nn.MaxPool3d(kernel_size=(2, 2, 2), stride=(2, 2, 2))

        self.conv5a = nn.Conv3d(512, 512, kernel_size=(3, 3, 3), padding=(1, 1, 1))
        self.conv5b = nn.Conv3d(512, 512, kernel_size=(3, 3, 3), padding=(1, 1, 1))
        self.pool5 = nn.MaxPool3d(kernel_size=(2, 2, 2), stride=(2, 2, 2), padding=(0, 1, 1))

        self.fc6 = nn.Linear(8192, 4096)
        self.fc7 = nn.Linear(4096, 4096)
        self.fc8 = nn.Linear(4096, num_classes)

        self.dropout = nn.Dropout(p=0.5)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.relu(self.conv1(x))
        x = self.pool1(x)
        x = self.relu(self.conv2(x))
        x = self.pool2(x)
        x = self.relu(self.conv3a(x))
        x = self.relu(self.conv3b(x))
        x = self.pool3(x)
        x = self.relu(self.conv4a(x))
        x = self.relu(self.conv4b(x))
        x = self.pool4(x)
        x = self.relu(self.conv5a(x))
        x = self.relu(self.conv5b(x))
        x = self.pool5(x)
        x = x.view(-1, 8192)
        x = self.relu(self.fc6(x))
        x = self.dropout(x)
        x = self.relu(self.fc7(x))
        x = self.dropout(x)
        logits = self.fc8(x)
        return logits


class VideoPredictor:
    def __init__(self, model_path, num_classes=101, class_names_file=None):
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.num_classes = num_classes

        # 加载模型
        self.model = C3D(num_classes)

        # 加载训练好的权重
        checkpoint = torch.load(model_path, map_location=self.device)

        # 处理多GPU训练保存的模型
        if isinstance(checkpoint, dict) and 'state_dict' in checkpoint:
            state_dict = checkpoint['state_dict']
        else:
            state_dict = checkpoint

        # 移除module前缀（如果使用DataParallel训练）
        new_state_dict = {}
        for k, v in state_dict.items():
            if k.startswith('module.'):
                new_state_dict[k[7:]] = v
            else:
                new_state_dict[k] = v

        self.model.load_state_dict(new_state_dict)
        self.model.to(self.device)
        self.model.eval()

        # 加载类别名称
        self.class_names = self.load_class_names(class_names_file)

    def load_class_names(self, class_names_file):
        """加载类别名称"""
        if class_names_file and os.path.exists(class_names_file):
            class_names = {}
            with open(class_names_file, 'r') as f:
                for line in f:
                    idx, name = line.strip().split(' ')
                    class_names[int(idx) - 1] = name
            return class_names
        else:
            # 如果没有类别文件，返回数字索引
            return {i: f"Class_{i}" for i in range(self.num_classes)}

    def preprocess_video(self, video_path, num_frames=16):
        """预处理视频，提取帧并进行预处理"""
        cap = cv2.VideoCapture(video_path)
        frames = []

        # 读取所有帧
        while True:
            ret, frame = cap.read()
            if not ret:
                break
            frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            frame_resized = cv2.resize(frame_rgb, (112, 112))
            frame_normalized = frame_resized.astype(np.float32)
            frame_chw = np.transpose(frame_normalized, (2, 0, 1))
            frames.append(frame_chw)

        cap.release()

        if len(frames) == 0:
            raise ValueError(f"无法从视频 {video_path} 中读取帧")

        # 采样策略：从视频中均匀采样num_frames帧
        total_frames = len(frames)
        if total_frames >= num_frames:
            # 均匀采样
            indices = np.linspace(0, total_frames - 1, num_frames, dtype=int)
            sampled_frames = [frames[i] for i in indices]
        else:
            # 如果帧数不足，重复最后一帧
            sampled_frames = frames
            while len(sampled_frames) < num_frames:
                sampled_frames.append(frames[-1])

        # 转换为tensor
        frames_tensor = torch.tensor(np.stack(sampled_frames), dtype=torch.float32)
        frames_tensor = frames_tensor.unsqueeze(0)  # 添加batch维度
        frames_tensor = frames_tensor.transpose(1, 2)  # (B, T, C, H, W) -> (B, C, T, H, W)

        return frames_tensor

    def predict(self, video_path):
        """对单个视频进行预测"""
        try:
            # 预处理视频
            input_tensor = self.preprocess_video(video_path)
            input_tensor = input_tensor.to(self.device)

            # 预测
            with torch.no_grad():
                outputs = self.model(input_tensor)
                probabilities = torch.softmax(outputs, dim=1)
                predicted_class = torch.argmax(outputs, dim=1).item()
                confidence = probabilities[0][predicted_class].item()

            # 获取类别名称
            class_name = self.class_names.get(predicted_class, f"Class_{predicted_class}")

            # 获取top-3预测结果
            top3_probs, top3_indices = torch.topk(probabilities, 3)
            top3_results = []
            for i in range(3):
                idx = top3_indices[0][i].item()
                prob = top3_probs[0][i].item()
                name = self.class_names.get(idx, f"Class_{idx}")
                top3_results.append((name, prob))

            return {
                'predicted_class': predicted_class,
                'class_name': class_name,
                'confidence': confidence,
                'top3_predictions': top3_results,
                'all_probabilities': probabilities.cpu().numpy()[0]
            }

        except Exception as e:
            return {'error': str(e)}

    def predict_batch(self, video_dir):
        """批量预测目录中的所有视频"""
        video_extensions = ['.avi', '.mp4', '.mov', '.mkv']
        video_files = []

        for ext in video_extensions:
            video_files.extend(Path(video_dir).glob(f'*{ext}'))
            video_files.extend(Path(video_dir).glob(f'*{ext.upper()}'))

        results = {}
        for video_file in video_files:
            print(f"处理视频: {video_file.name}")
            result = self.predict(str(video_file))
            results[video_file.name] = result

        return results

    def save_prediction_video(self, video_path, output_path):
        """保存带预测结果的视频"""
        try:
            # 先进行一次预测获取结果
            result = self.predict(video_path)
            if 'error' in result:
                print(f"预测错误: {result['error']}")
                return None

            # 打开视频文件
            cap = cv2.VideoCapture(video_path)

            # 获取视频信息
            fps = cap.get(cv2.CAP_PROP_FPS)
            width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
            height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
            total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

            # 设置视频写入器
            fourcc = cv2.VideoWriter_fourcc(*'XVID')
            out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))

            print(f"视频信息: {width}x{height}, {fps}fps, 总帧数: {total_frames}")
            print(f"预测结果: {result['class_name']} (置信度: {result['confidence']:.4f})")

            frame_count = 0
            while True:
                ret, frame = cap.read()
                if not ret:
                    break

                frame_count += 1

                # 在帧上绘制预测结果
                cv2.putText(frame, f"Prediction: {result['class_name']}", (20, 30),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
                cv2.putText(frame, f"Confidence: {result['confidence']:.4f}", (20, 60),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
                cv2.putText(frame, f"Frame: {frame_count}/{total_frames}", (20, 90),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)

                # 绘制Top-3预测结果
                for i, (name, prob) in enumerate(result['top3_predictions']):
                    y_pos = 120 + i * 30
                    color = (0, 255, 0) if i == 0 else (255, 255, 0)
                    cv2.putText(frame, f"{i + 1}. {name}: {prob:.4f}", (20, y_pos),
                                cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 1)

                # 写入帧
                out.write(frame)

                # 显示进度
                if frame_count % 50 == 0:
                    print(f"处理进度: {frame_count}/{total_frames} 帧")

            # 释放资源
            cap.release()
            out.release()

            print(f"带预测结果的视频已保存到: {output_path}")
            return result

        except Exception as e:
            print(f"保存预测视频错误: {str(e)}")
            return {'error': str(e)}


def main():
    parser = argparse.ArgumentParser(description='C3D视频分类预测')
    parser.add_argument('--model_path', type=str, required=True, help='训练好的模型路径')
    parser.add_argument('--video_path', type=str, required=True, help='要预测的视频路径或视频目录')
    parser.add_argument('--class_names', type=str, default=None, help='类别名称文件路径')
    parser.add_argument('--num_classes', type=int, default=101, help='类别数量')
    parser.add_argument('--output_video', type=str, default=None, help='输出视频路径')

    args = parser.parse_args()

    # 初始化预测器
    predictor = VideoPredictor(
        model_path=args.model_path,
        num_classes=args.num_classes,
        class_names_file=args.class_names
    )

    # 判断输入是文件还是目录
    if os.path.isfile(args.video_path):
        if args.output_video:
            # 保存带预测结果的视频
            result = predictor.save_prediction_video(
                args.video_path,
                args.output_video
            )
        else:
            # 单个视频预测
            result = predictor.predict(args.video_path)

        if result and 'error' not in result:
            print(f"\n预测结果:")
            print(f"视频文件: {os.path.basename(args.video_path)}")
            print(f"预测类别: {result['class_name']} (ID: {result['predicted_class']})")
            print(f"置信度: {result['confidence']:.4f}")
            print(f"\nTop-3预测:")
            for i, (name, prob) in enumerate(result['top3_predictions']):
                print(f"  {i + 1}. {name}: {prob:.4f}")
        elif result and 'error' in result:
            print(f"预测错误: {result['error']}")

    elif os.path.isdir(args.video_path):
        # 批量预测
        results = predictor.predict_batch(args.video_path)

        print(f"\n批量预测结果:")
        for video_name, result in results.items():
            if 'error' in result:
                print(f"{video_name}: 错误 - {result['error']}")
            else:
                print(f"{video_name}: {result['class_name']} (置信度: {result['confidence']:.4f})")

    else:
        print(f"错误: {args.video_path} 不是有效的文件或目录")


if __name__ == "__main__":
    # 如果没有命令行参数，可以使用以下代码直接测试
    # main()

    # 或者直接使用以下代码进行测试
    model_path = "./checkpoints_shop/acc/best_acc.pth"  # 修改为你的模型路径
    video_path = "/mnt/Data/daihongliang/便利店数据集/drinking/10月3日 (1)141556-1.mp4"  # 修改为你的视频路径
    class_names_file = "/mnt/Data/daihongliang/shopTrainTestlist/classshop.txt"  # 修改为你的类别文件路径
    output_video_path = "output_with_predictions.avi"  # 保存结果视频

    if os.path.exists(model_path):
        predictor = VideoPredictor(
            model_path=model_path,
            num_classes=4,
            class_names_file=class_names_file
        )

        # 方法1: 简单预测（只输出文本结果）
        print("=== 简单预测 ===")
        result = predictor.predict(video_path)

        if 'error' in result:
            print(f"预测错误: {result['error']}")
        else:
            print(f"\n预测结果:")
            print(f"视频文件: {os.path.basename(video_path)}")
            print(f"预测类别: {result['class_name']} (ID: {result['predicted_class']})")
            print(f"置信度: {result['confidence']:.4f}")
            print(f"\nTop-3预测:")
            for i, (name, prob) in enumerate(result['top3_predictions']):
                print(f"  {i + 1}. {name}: {prob:.4f}")

        # 方法2: 保存带预测结果的视频
        print("\n=== 保存预测视频 ===")
        result = predictor.save_prediction_video(
            video_path,
            output_video_path
        )

    else:
        print(f"模型文件不存在: {model_path}")

=== 简单预测 ===

预测结果:
视频文件: 10月3日 (1)141556-1.mp4
预测类别: hiding (ID: 2)
置信度: 0.5954

Top-3预测:
  1. hiding: 0.5954
  2. drinking: 0.2383
  3. eating: 0.0947

=== 保存预测视频 ===
视频信息: 1920x1080, 30.0fps, 总帧数: 124
预测结果: hiding (置信度: 0.5954)
处理进度: 50/124 帧
处理进度: 100/124 帧
带预测结果的视频已保存到: output_with_predictions.avi
